In [ ]:
!pip install torchmetrics datasets

In [ ]:
import torch
import torchmetrics
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


from sklearn.model_selection import train_test_split

import datasets
from datasets import load_dataset

import re
import pandas as pd
import seaborn as sns
from collections import Counter
import matplotlib.pyplot as plt

# Data Handling

In [ ]:
dataset = load_dataset("codesignal/sms-spam-collection")
df = pd.DataFrame(dataset["train"])

df.head()

In [ ]:
# Encode labels
df["label"] = df["label"].map({"ham": 0, "spam": 1})

df.head()

In [ ]:
# Split data into train, val, test
train_df, temp_df = train_test_split(
    df, test_size=0.3, stratify=df["label"], random_state=42
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42
)

print(len(train_df), len(val_df), len(test_df))

## Tokenization Strategy

In [ ]:
def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    tokens = text.split()
    return tokens

## Vocabulary Building

In [ ]:
max_vocab_size = 5000

counter = Counter()

for text in train_df["message"]:
    tokens = tokenize(text)
    counter.update(tokens)

most_common = counter.most_common(max_vocab_size - 2)

vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

for i, (word, _) in enumerate(most_common, start=2):
    vocab[word] = i

vocab_size = len(vocab)

print("Vocab Size: ", vocab_size)

## Text to Sequence Encoding

In [ ]:
def encode(tokens, vocab):
    return [vocab.get(token, vocab["<UNK>"]) for token in tokens]

## Padding Function

In [ ]:
max_length = 40

def pad_sequence(seq, max_lenght):
    if len(seq) < max_length:
        seq = seq + [0] * (max_length - len(seq))
    else:
        seq = seq[:max_length]
    
    return seq

# Dataset Class

In [ ]:
class SpamDataset(Dataset):
    def __init__(self, dataframe, vocab, max_length):
        self.texts = dataframe["message"].tolist()
        self.labels = dataframe["label"].tolist()
        self.vocab = vocab
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, index):
        text = self.texts[index]
        label = self.labels[index]

        tokens = tokenize(text)
        encoded = encode(tokens, self.vocab)
        padded = pad_sequence(encoded, self.max_length)

        return (
            torch.tensor(padded, dtype=torch.long),
            torch.tensor(label, dtype=torch.float)
        )

# Helper Functions

In [ ]:
def get_loaders(train_df, val_df, test_df, vocab, max_length=40, batch_size=32):
    
    train_dataset = SpamDataset(train_df, vocab, max_length)
    val_dataset = SpamDataset(val_df, vocab, max_length)
    test_dataset = SpamDataset(test_df, vocab, max_length)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

In [ ]:
def training_setup(model):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    return criterion, optimizer